In [1]:
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from itertools import permutations
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori 
from mlxtend.frequent_patterns import association_rules
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df=pd.read_csv("Megastore_Dataset_Task_3 3.csv")
df

,OrderID,ProductName,Quantity,InvoiceDate,UnitPrice,TotalCost,Country,DiscountApplied,OrderPriority,Region,Segment,ExpeditedShipping,PaymentMethod,CustomerOrderSatisfaction
0,536370,INFLATABLE POLITICAL GLOBE,48,12/1/10 8:45,$0.85,$40.80,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
1,536370,SET2 RED RETROSPOT TEA TOWELS,18,12/1/10 8:45,$2.95,$53.10,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
2,536370,PANDA AND BUNNIES STICKER SHEET,12,12/1/10 8:45,$0.85,$10.20,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
3,536370,RED TOADSTOOL LED NIGHT LIGHT,24,12/1/10 8:45,$1.65,$39.60,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
4,536370,VINTAGE HEADS AND TAILS CARD GAME,24,12/1/10 8:45,$1.25,$30.00,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8229,580263,DANISH ROSE DELUXE COASTER,12,12/2/11 12:43,$0.85,$10.20,United States,No,High,Southeast,Corporate,Yes,PayPal,Dissatisfied
8230,580263,CACTI TLIGHT CANDLES,16,12/2/11 12:43,$0.42,$6.72,United States,No,High,Southeast,Corporate,Yes,PayPal,Dissatisfied
8231,581316,RED RETROSPOT SUGAR JAM BOWL,1,12/8/11 11:46,$2.55,$2.55,United States,No,High,Southeast,Consumer,No,Credit Card,Very Satisfied
8232,581316,GLASS SONGBIRD STORAGE JAR,1,12/8/11 11:46,$12.50,$12.50,United States,No,High,Southeast,Consumer,No,Credit Card,Very Satisfied


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8234 entries, 0 to 8233
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   OrderID                    8234 non-null   int64 
 1   ProductName                8234 non-null   object
 2   Quantity                   8234 non-null   int64 
 3   InvoiceDate                8234 non-null   object
 4   UnitPrice                  8234 non-null   object
 5   TotalCost                  8234 non-null   object
 6   Country                    8234 non-null   object
 7   DiscountApplied            8234 non-null   object
 8   OrderPriority              8234 non-null   object
 9   Region                     8234 non-null   object
 10  Segment                    8234 non-null   object
 11  ExpeditedShipping          8234 non-null   object
 12  PaymentMethod              8234 non-null   object
 13  CustomerOrderSatisfaction  8234 non-null   object
dtypes: int64

In [7]:
df.isnull().any()

OrderID                      False
ProductName                  False
Quantity                     False
InvoiceDate                  False
UnitPrice                    False
TotalCost                    False
Country                      False
DiscountApplied              False
OrderPriority                False
Region                       False
Segment                      False
ExpeditedShipping            False
PaymentMethod                False
CustomerOrderSatisfaction    False
dtype: bool

In [9]:
df.isna().any()

OrderID                      False
ProductName                  False
Quantity                     False
InvoiceDate                  False
UnitPrice                    False
TotalCost                    False
Country                      False
DiscountApplied              False
OrderPriority                False
Region                       False
Segment                      False
ExpeditedShipping            False
PaymentMethod                False
CustomerOrderSatisfaction    False
dtype: bool

In [11]:
df.describe()

,OrderID,Quantity
count,8234.000000,8234.000000
mean,560874.506923,13.705125
std,13082.500625,21.494536
min,536370.000000,1.000000
25%,549274.000000,6.000000
50%,563502.000000,10.000000
75%,571864.000000,12.000000
max,581587.000000,912.000000


In [13]:
df.nunique()

OrderID                       441
ProductName                  1562
Quantity                       57
InvoiceDate                   441
UnitPrice                      86
TotalCost                     443
Country                         1
DiscountApplied                 2
OrderPriority                   2
Region                          2
Segment                         2
ExpeditedShipping               2
PaymentMethod                   2
CustomerOrderSatisfaction       5
dtype: int64

In [15]:
unique1 = df['CustomerOrderSatisfaction'].unique()
print("Unique values using unique():", unique1)

Unique values using unique(): ['Satisfied' 'Dissatisfied' 'Prefer not to answer' 'Very Satisfied'
 'Very Dissatisfied']


In [17]:
df.head()

,OrderID,ProductName,Quantity,InvoiceDate,UnitPrice,TotalCost,Country,DiscountApplied,OrderPriority,Region,Segment,ExpeditedShipping,PaymentMethod,CustomerOrderSatisfaction
0,536370,INFLATABLE POLITICAL GLOBE,48,12/1/10 8:45,$0.85,$40.80,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
1,536370,SET2 RED RETROSPOT TEA TOWELS,18,12/1/10 8:45,$2.95,$53.10,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
2,536370,PANDA AND BUNNIES STICKER SHEET,12,12/1/10 8:45,$0.85,$10.20,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
3,536370,RED TOADSTOOL LED NIGHT LIGHT,24,12/1/10 8:45,$1.65,$39.60,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied
4,536370,VINTAGE HEADS AND TAILS CARD GAME,24,12/1/10 8:45,$1.25,$30.00,United States,Yes,High,Northeast,Corporate,Yes,Credit Card,Satisfied


In [19]:
order1 = df['OrderID'].value_counts().idxmax()
transaction1 = df[df['OrderID'] == order1][['OrderID', 'ProductName', 'Quantity']]
print(transaction1.head(10))

      OrderID                       ProductName  Quantity
5731   570672              RED RETROSPOT PURSE          6
5732   570672               GREEN POLKADOT BOWL        16
5733   570672                BLUE POLKADOT BOWL         8
5734   570672                RED RETROSPOT BOWL         8
5735   570672                PINK POLKADOT BOWL         8
5736   570672  RED RETROSPOT CHILDRENS UMBRELLA         6
5737   570672                 MR ROBOT SOFT TOY         2
5738   570672            STRAWBERRY SHOPPER BAG        10
5739   570672         RED RETROSPOT SHOPPER BAG        10
5740   570672            WOODLAND CHARLOTTE BAG        10


In [21]:
transactions = df.groupby('OrderID')['ProductName'].apply(list).values.tolist()
transactions

[['INFLATABLE POLITICAL GLOBE ',
  'SET2 RED RETROSPOT TEA TOWELS ',
  'PANDA AND BUNNIES STICKER SHEET',
  'RED TOADSTOOL LED NIGHT LIGHT',
  'VINTAGE HEADS AND TAILS CARD GAME ',
  'STARS GIFT TAPE ',
  'VINTAGE SEASIDE JIGSAW PUZZLES',
  'ROUND SNACK BOXES SET OF4 WOODLAND ',
  'MINI PAINT SET VINTAGE ',
  'MINI JIGSAW CIRCUS PARADE ',
  'MINI JIGSAW SPACEBOY',
  'SPACEBOY LUNCH BOX ',
  'CIRCUS PARADE LUNCH BOX ',
  'LUNCH BOX I LOVE LONDON',
  'CHARLOTTE BAG DOLLY GIRL DESIGN',
  'ALARM CLOCK BAKELIKE GREEN',
  'ALARM CLOCK BAKELIKE RED ',
  'ALARM CLOCK BAKELIKE PINK',
  'SET 2 TEA TOWELS I LOVE LONDON '],
 ['POLKADOT RAIN HAT ',
  'VINTAGE HEADS AND TAILS CARD GAME ',
  'MINI JIGSAW DOLLY GIRL',
  'MINI JIGSAW SPACEBOY',
  'PICTURE DOMINOES',
  'CHARLOTTE BAG DOLLY GIRL DESIGN'],
 ['EDWARDIAN PARASOL RED',
  'LUNCH BAG RED RETROSPOT',
  'LUNCH BAG WOODLAND',
  'ASSORTED COLOUR MINI CASES',
  'RED RETROSPOT MINI CASES',
  'BIG DOUGHNUT FRIDGE MAGNETS',
  'RED  HARMONICA IN BOX ',

In [23]:
print(transactions[0])

['INFLATABLE POLITICAL GLOBE ', 'SET2 RED RETROSPOT TEA TOWELS ', 'PANDA AND BUNNIES STICKER SHEET', 'RED TOADSTOOL LED NIGHT LIGHT', 'VINTAGE HEADS AND TAILS CARD GAME ', 'STARS GIFT TAPE ', 'VINTAGE SEASIDE JIGSAW PUZZLES', 'ROUND SNACK BOXES SET OF4 WOODLAND ', 'MINI PAINT SET VINTAGE ', 'MINI JIGSAW CIRCUS PARADE ', 'MINI JIGSAW SPACEBOY', 'SPACEBOY LUNCH BOX ', 'CIRCUS PARADE LUNCH BOX ', 'LUNCH BOX I LOVE LONDON', 'CHARLOTTE BAG DOLLY GIRL DESIGN', 'ALARM CLOCK BAKELIKE GREEN', 'ALARM CLOCK BAKELIKE RED ', 'ALARM CLOCK BAKELIKE PINK', 'SET 2 TEA TOWELS I LOVE LONDON ']


In [25]:
encoder = TransactionEncoder()
encoded_transactions = encoder.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(encoded_transactions, columns= encoder.columns_)
df_encoded

,50S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,NINE DRAWER OFFICE TIDY,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,TRELLIS COAT RACK,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,...,WRAP VINTAGE PETALS DESIGN,YELLOW COAT RACK PARIS FASHION,YELLOW GIANT GARDEN THERMOMETER,YELLOW SHARK HELICOPTER,ZINC STAR TLIGHT HOLDER,ZINC FOLKART SLEIGH BELLS,ZINC HERB GARDEN CONTAINER,ZINC METAL HEART DECORATION,ZINC TLIGHT HOLDER STAR LARGE,ZINC TLIGHT HOLDER STARS SMALL
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
437,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
438,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
439,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [27]:
frequent_itemsets = apriori(df_encoded, use_colnames= True, min_support= 0.01)
frequent_itemsets

,support,itemsets
0,0.020408,( DOLLY GIRL BEAKER)
1,0.011338,( I LOVE LONDON MINI BACKPACK)
2,0.013605,( SET 2 TEA TOWELS I LOVE LONDON )
3,0.036281,( SPACEBOY BABY GIFT SET)
4,0.027211,(10 COLOUR SPACEBOY PEN)
...,...,...
8335,0.011338,"(PACK OF 20 SKULL PAPER NAPKINS, SET6 RED SPOT..."
8336,0.011338,"(PACK OF 20 SKULL PAPER NAPKINS, SET6 RED SPOT..."
8337,0.011338,"(SET6 RED SPOTTY PAPER PLATES, SET OF 9 HEART ..."
8338,0.011338,"(SET6 RED SPOTTY PAPER PLATES, PLASTERS IN TIN..."


In [29]:
print(len(frequent_itemsets))

8340


In [31]:
print(frequent_itemsets.head())

    support                            itemsets
0  0.020408                ( DOLLY GIRL BEAKER)
1  0.011338      ( I LOVE LONDON MINI BACKPACK)
2  0.013605  ( SET 2 TEA TOWELS I LOVE LONDON )
3  0.036281           ( SPACEBOY BABY GIFT SET)
4  0.027211            (10 COLOUR SPACEBOY PEN)


In [33]:
rules = association_rules(frequent_itemsets, metric = "support", min_threshold = 0.01)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(CHARLOTTE BAG DOLLY GIRL DESIGN),( DOLLY GIRL BEAKER),0.058957,0.020408,0.011338,0.192308,9.423077,1.0,0.010135,1.212828,0.949880,0.166667,0.175481,0.373932
1,( DOLLY GIRL BEAKER),(CHARLOTTE BAG DOLLY GIRL DESIGN),0.020408,0.058957,0.011338,0.555556,9.423077,1.0,0.010135,2.117347,0.912500,0.166667,0.527711,0.373932
2,(DOLLY GIRL CHILDRENS BOWL),( DOLLY GIRL BEAKER),0.040816,0.020408,0.015873,0.388889,19.055556,1.0,0.015040,1.602968,0.987842,0.350000,0.376157,0.583333
3,( DOLLY GIRL BEAKER),(DOLLY GIRL CHILDRENS BOWL),0.020408,0.040816,0.015873,0.777778,19.055556,1.0,0.015040,4.316327,0.967262,0.350000,0.768322,0.583333
4,(DOLLY GIRL CHILDRENS CUP),( DOLLY GIRL BEAKER),0.036281,0.020408,0.013605,0.375000,18.375000,1.0,0.012865,1.567347,0.981176,0.315789,0.361979,0.520833
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85217,(SET6 RED SPOTTY PAPER CUPS),"(PACK OF 20 SKULL PAPER NAPKINS, SET6 RED SPOT...",0.122449,0.011338,0.011338,0.092593,8.166667,1.0,0.009950,1.089546,1.000000,0.092593,0.082187,0.546296
85218,(SET OF 9 BLACK SKULL BALLOONS),"(PACK OF 20 SKULL PAPER NAPKINS, SET6 RED SPOT...",0.058957,0.013605,0.011338,0.192308,14.134615,1.0,0.010536,1.221250,0.987470,0.185185,0.181167,0.512821
85219,(PACK OF 6 SKULL PAPER CUPS),"(PACK OF 20 SKULL PAPER NAPKINS, SET6 RED SPOT...",0.056689,0.011338,0.011338,0.200000,17.640000,1.0,0.010695,1.235828,1.000000,0.200000,0.190826,0.600000
85220,(SET20 RED RETROSPOT PAPER NAPKINS ),"(PACK OF 20 SKULL PAPER NAPKINS, SET6 RED SPOT...",0.117914,0.011338,0.011338,0.096154,8.480769,1.0,0.010001,1.093839,1.000000,0.096154,0.085789,0.548077


In [35]:
print(rules)

                                antecedents  \
0         (CHARLOTTE BAG DOLLY GIRL DESIGN)   
1                      ( DOLLY GIRL BEAKER)   
2               (DOLLY GIRL CHILDRENS BOWL)   
3                      ( DOLLY GIRL BEAKER)   
4                (DOLLY GIRL CHILDRENS CUP)   
...                                     ...   
85217          (SET6 RED SPOTTY PAPER CUPS)   
85218       (SET OF 9 BLACK SKULL BALLOONS)   
85219          (PACK OF 6 SKULL PAPER CUPS)   
85220  (SET20 RED RETROSPOT PAPER NAPKINS )   
85221        (PACK OF 6 SKULL PAPER PLATES)   

                                             consequents  antecedent support  \
0                                   ( DOLLY GIRL BEAKER)            0.058957   
1                      (CHARLOTTE BAG DOLLY GIRL DESIGN)            0.020408   
2                                   ( DOLLY GIRL BEAKER)            0.040816   
3                            (DOLLY GIRL CHILDRENS BOWL)            0.020408   
4                                  

In [37]:
print(rules.columns)

Index(['antecedents', 'consequents', 'antecedent support',
       'consequent support', 'support', 'confidence', 'lift',
       'representativity', 'leverage', 'conviction', 'zhangs_metric',
       'jaccard', 'certainty', 'kulczynski'],
      dtype='object')


In [39]:
print(rules[['antecedents', 'consequents']])

                                antecedents  \
0         (CHARLOTTE BAG DOLLY GIRL DESIGN)   
1                      ( DOLLY GIRL BEAKER)   
2               (DOLLY GIRL CHILDRENS BOWL)   
3                      ( DOLLY GIRL BEAKER)   
4                (DOLLY GIRL CHILDRENS CUP)   
...                                     ...   
85217          (SET6 RED SPOTTY PAPER CUPS)   
85218       (SET OF 9 BLACK SKULL BALLOONS)   
85219          (PACK OF 6 SKULL PAPER CUPS)   
85220  (SET20 RED RETROSPOT PAPER NAPKINS )   
85221        (PACK OF 6 SKULL PAPER PLATES)   

                                             consequents  
0                                   ( DOLLY GIRL BEAKER)  
1                      (CHARLOTTE BAG DOLLY GIRL DESIGN)  
2                                   ( DOLLY GIRL BEAKER)  
3                            (DOLLY GIRL CHILDRENS BOWL)  
4                                   ( DOLLY GIRL BEAKER)  
...                                                  ...  
85217  (PACK OF 20 SKU

In [41]:
ordinal_mapping = {
    "OrderPriority": ["Medium", "High"],
    "CustomerOrderSatisfaction": [ "Very Dissatisfied", "Dissatisfied", "Prefer not to answer",
        "Satisfied", "Very Satisfied" ]}

In [43]:
encoder2 = OrdinalEncoder(categories=[
    ordinal_mapping["OrderPriority"],
    ordinal_mapping["CustomerOrderSatisfaction"]
])
df[["OrderPriority", "CustomerOrderSatisfaction"]] = encoder2.fit_transform(
    df[["OrderPriority", "CustomerOrderSatisfaction"]])

In [45]:
df2 = pd.get_dummies(df, columns=["Country", "PaymentMethod"])
basket = df2.groupby(['OrderID', 'ProductName'])['Quantity'].sum().unstack().fillna(0)
basket = basket.map(lambda x: 1 if x > 0 else 0)
basket

ProductName,50S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,NINE DRAWER OFFICE TIDY,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,TRELLIS COAT RACK,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,...,WRAP VINTAGE PETALS DESIGN,YELLOW COAT RACK PARIS FASHION,YELLOW GIANT GARDEN THERMOMETER,YELLOW SHARK HELICOPTER,ZINC STAR TLIGHT HOLDER,ZINC FOLKART SLEIGH BELLS,ZINC HERB GARDEN CONTAINER,ZINC METAL HEART DECORATION,ZINC TLIGHT HOLDER STAR LARGE,ZINC TLIGHT HOLDER STARS SMALL
OrderID,,,,,,,,,,,,,,,,,,,,,
536370,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536852,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
536974,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
537065,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
537463,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
581001,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581171,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
581279,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [47]:
freq = apriori(basket, min_support=0.01, use_colnames=True)
rulesmba = association_rules(freq, metric="lift", min_threshold=0.5)

/opt/anaconda3/lib/python3.12/site-packages/mlxtend/frequent_patterns/fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [49]:
top_rules = rulesmba.sort_values(by=['confidence', 'lift', 'support'], ascending=False).head(3)
top_rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
4800,(SMALL MARSHMALLOWS PINK BOWL),(SMALL DOLLY MIX DESIGN ORANGE BOWL),0.011338,0.011338,0.011338,1.0,88.2,1.0,0.011209,inf,1.0,1.0,1.0,1.0
4801,(SMALL DOLLY MIX DESIGN ORANGE BOWL),(SMALL MARSHMALLOWS PINK BOWL),0.011338,0.011338,0.011338,1.0,88.2,1.0,0.011209,inf,1.0,1.0,1.0,1.0
53691,"(CHARLOTTE BAG DOLLY GIRL DESIGN, PLASTERS IN ...","(PLASTERS IN TIN SPACEBOY, ALARM CLOCK BAKELIK...",0.011338,0.011338,0.011338,1.0,88.2,1.0,0.011209,inf,1.0,1.0,1.0,1.0


In [51]:
basket.to_csv("transactional_megastore_dataset.csv")
df2.to_csv("Cleaned_Megastore_Data.csv", index=False)

In [53]:
onehot = encoder.transform(transactions)
onehot = pd.DataFrame(onehot, columns = encoder.columns_)
onehot

,50S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,NINE DRAWER OFFICE TIDY,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,TRELLIS COAT RACK,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 EGG HOUSE PAINTED WOOD,...,WRAP VINTAGE PETALS DESIGN,YELLOW COAT RACK PARIS FASHION,YELLOW GIANT GARDEN THERMOMETER,YELLOW SHARK HELICOPTER,ZINC STAR TLIGHT HOLDER,ZINC FOLKART SLEIGH BELLS,ZINC HERB GARDEN CONTAINER,ZINC METAL HEART DECORATION,ZINC TLIGHT HOLDER STAR LARGE,ZINC TLIGHT HOLDER STARS SMALL
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
436,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
437,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
438,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
439,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [55]:
print(onehot.mean())

 50S CHRISTMAS GIFT BAG LARGE       0.002268
 DOLLY GIRL BEAKER                  0.020408
 I LOVE LONDON MINI BACKPACK        0.011338
 NINE DRAWER OFFICE TIDY            0.002268
 SET 2 TEA TOWELS I LOVE LONDON     0.013605
                                      ...   
ZINC FOLKART SLEIGH BELLS           0.018141
ZINC HERB GARDEN CONTAINER          0.002268
ZINC METAL HEART DECORATION         0.009070
ZINC TLIGHT HOLDER STAR LARGE       0.002268
ZINC TLIGHT HOLDER STARS SMALL      0.004535
Length: 1562, dtype: float64


In [57]:
bag_headers = [i for i in onehot.columns if i.lower().find('bag')>=0]
dolly_headers = [i for i in onehot.columns if i.lower().find('dolly')>=0]
bags = onehot[bag_headers]
dollys = onehot[dolly_headers]
print(bags)

      50S CHRISTMAS GIFT BAG LARGE  AIRLINE BAG VINTAGE JET SET BROWN  \
0                            False                              False   
1                            False                              False   
2                            False                              False   
3                            False                              False   
4                            False                              False   
..                             ...                                ...   
436                          False                              False   
437                          False                              False   
438                          False                              False   
439                          False                              False   
440                          False                              False   

     AIRLINE BAG VINTAGE JET SET RED  AIRLINE BAG VINTAGE JET SET WHITE  \
0                              False            

In [59]:
bags = (bags.sum(axis=1) > 0.0).values
dollys = (dollys.sum(axis=1) > 0.0).values
print(bags)

[ True  True  True  True  True  True False False  True False  True  True
  True  True  True  True False  True False  True  True False False False
 False  True  True  True  True  True  True False  True False False False
  True False False False  True  True  True  True False  True  True False
 False False  True False False False False False  True  True False False
  True  True  True  True False  True False False False  True  True  True
 False False False False False False False False False False  True False
  True False  True False False  True False False  True False  True False
  True False False  True  True False  True False False False  True  True
 False  True  True False  True False False False  True  True False  True
  True  True False  True False  True False False False  True  True False
 False False  True  True  True False False False  True  True  True  True
 False  True False False False  True  True  True False  True  True  True
 False  True False  True  True False  True  True  T

In [61]:
aggregated = pd.DataFrame(np.vstack([bags, dollys]).T, columns = ['bags', 'dolly'])
print(aggregated.head())

   bags  dolly
0  True   True
1  True   True
2  True  False
3  True   True
4  True  False


In [63]:
print(aggregated.mean())

bags     0.517007
dolly    0.333333
dtype: float64
